# Module 7: A Systemic Yankees Development Problem

Volpe and Dominguez aren't isolated cases. The Yankees system has repeatedly produced hitters who:
1. Show elite discipline metrics in the minors
2. Win org-level "best strike zone discipline" awards
3. Regression against MLB-quality pitching

This isn't bad luck. It's a pattern — and it points to a systemic failure in how the Yankees development pipeline prepares hitters for the jump to MLB.

We compare Yankees-developed hitters against prospects from other organizations to see if the discipline regression is uniquely severe.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="urllib3")

import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from fire_fishman.data.statcast import get_statcast_pitches
from fire_fishman.data.prospects import (
    get_yankees_system_df, get_prospect_df, get_org_prospects, get_org_summary,
    MILB_STATS, TARGET_ORGS, ELITE_DEV_ORGS, PROSPECT_DATA,
)
from fire_fishman.features.pitch_level import compute_plate_discipline, compute_whiff_by_pitch_type

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 7)

pitches_2023 = get_statcast_pitches(2023)
pitches_2024 = get_statcast_pitches(2024)
all_pitches = pd.concat([pitches_2023, pitches_2024])


## 1. The Evidence: MiLB Discipline Awards → MLB Regression

Both Volpe and Dominguez won Baseball America's "Best Strike-Zone Discipline" for the Yankees system. Both regressed.

In [ ]:
print("MINOR LEAGUE TRACK RECORDS")
print("=" * 70)

for name, seasons in MILB_STATS.items():
    print(f"\n  {name}:")
    for year, stats in seasons.items():
        line = f"    {year} ({stats['level']}): .{int(stats.get('avg',0)*1000):03d}/.{int(stats.get('obp',0)*1000):03d}/.{int(stats.get('slg',0)*1000):03d}"
        if 'k_pct' in stats:
            line += f"  K%: {stats['k_pct']:.1%}"
        if 'bb_pct' in stats:
            line += f"  BB%: {stats['bb_pct']:.1%}"
        if 'note' in stats:
            line += f"\n           → {stats['note']}"
        print(line)

## 2. Yankees System vs Other Orgs: MLB Discipline Comparison

In [ ]:
# Compute discipline metrics for all prospects in target orgs
def compute_org_metrics(org, pitches_df):
    """Compute plate discipline metrics for all prospects from an org."""
    prospects = get_org_prospects(org)
    records = []
    for _, row in prospects.iterrows():
        bp = pitches_df[pitches_df["batter"] == row["mlbam_id"]]
        if len(bp) < 100:
            continue
        disc = compute_plate_discipline(bp, row["mlbam_id"])
        whiff = compute_whiff_by_pitch_type(bp, row["mlbam_id"])
        records.append({
            "name": row["name"],
            "org": org,
            "outcome": row["outcome"],
            "chase_rate": disc.get("chase_rate", np.nan),
            "brk_chase": whiff.get("chase_rate_breaking", np.nan),
            "off_chase": whiff.get("chase_rate_offspeed", np.nan),
            "whiff_rate": disc.get("whiff_rate", np.nan),
            "zone_contact": disc.get("zone_contact_rate", np.nan),
        })
    return pd.DataFrame(records)

# Build metrics for all target orgs
org_metrics = pd.concat([compute_org_metrics(org, all_pitches) for org in TARGET_ORGS],
                        ignore_index=True)

# Display per-org averages
print("DISCIPLINE METRICS BY DEVELOPMENT ORGANIZATION")
print("=" * 70)
metric_cols = ["chase_rate", "brk_chase", "off_chase", "whiff_rate", "zone_contact"]
for org in TARGET_ORGS:
    grp = org_metrics[org_metrics["org"] == org]
    if len(grp) == 0:
        continue
    print(f"\n  {org} (n={len(grp)}):")
    for _, row in grp.iterrows():
        print(f"    {row['name']:<25} Chase {row['chase_rate']:.1%}  BrkChase {row['brk_chase']:.1%}  ZoneCont {row['zone_contact']:.1%}")
    avgs = grp[metric_cols].mean()
    print(f"    {"AVG":<25} Chase {avgs['chase_rate']:.1%}  BrkChase {avgs['brk_chase']:.1%}  ZoneCont {avgs['zone_contact']:.1%}")


In [ ]:
# Visualization: Discipline by org (replacing Yankees vs Others)
org_colors = {"NYY": "#1a3c6e", "BAL": "#df4601", "CLE": "#e31937",
              "LAD": "#005a9c", "TB": "#092c5c", "ATL": "#ce1141"}

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
metrics = [("chase_rate", "Overall Chase Rate"),
           ("brk_chase", "Breaking Ball Chase Rate"),
           ("off_chase", "Offspeed Chase Rate")]

for ax, (col, title) in zip(axes, metrics):
    for i, org in enumerate(TARGET_ORGS):
        grp = org_metrics[org_metrics["org"] == org]
        vals = grp[col].dropna()
        jitter = np.random.RandomState(42).normal(0, 0.06, len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals,
                  s=100, color=org_colors.get(org, "gray"), alpha=0.7, zorder=5)
        if len(vals) > 0:
            ax.hlines(vals.mean(), i - 0.2, i + 0.2,
                     color=org_colors.get(org, "gray"), linewidth=3)
    ax.set_xticks(range(len(TARGET_ORGS)))
    ax.set_xticklabels(TARGET_ORGS, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.suptitle("MLB Plate Discipline by Development Organization\n"
             "Each dot = one prospect, horizontal line = org average",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/yankees_vs_orgs.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. The Pitch Mix Problem

Yankees hitters as a group see a different pitch mix than other teams' prospects.

In [ ]:
fb_codes = ["FF", "SI", "FC"]
brk_codes = ["SL", "CU", "KC", "SV", "SC"]
off_codes = ["CH", "FS", "FO"]

# Build the player groups in-cell (an earlier revision referenced
# yankees_players/other_orgs left over from a stale kernel).
prospect_df = get_prospect_df()
yankees_players = dict(
    prospect_df.loc[prospect_df["org"] == "NYY", ["name", "mlbam_id"]].values
)
other_orgs = dict(
    prospect_df.loc[prospect_df["org"] != "NYY", ["name", "mlbam_id"]].values
)

print("PITCH MIX SEEN BY EACH GROUP")
print("=" * 50)

group_mix = {}
for label, players in [("Yankees", yankees_players), ("Other Orgs", other_orgs)]:
    all_bp = all_pitches[all_pitches["batter"].isin(list(players.values()))]
    fb = all_bp["pitch_type"].isin(fb_codes).mean()
    brk = all_bp["pitch_type"].isin(brk_codes).mean()
    off = all_bp["pitch_type"].isin(off_codes).mean()
    group_mix[label] = {"fb": fb, "brk": brk, "off": off}
    print(f"  {label:<15} FB: {fb:.1%}  Breaking: {brk:.1%}  Offspeed: {off:.1%}")

league_fb = all_pitches["pitch_type"].isin(fb_codes).mean()
league_brk = all_pitches["pitch_type"].isin(brk_codes).mean()
league_off = all_pitches["pitch_type"].isin(off_codes).mean()
print(f"  {'League Avg':<15} FB: {league_fb:.1%}  Breaking: {league_brk:.1%}  Offspeed: {league_off:.1%}")

fb_diff = group_mix["Yankees"]["fb"] - group_mix["Other Orgs"]["fb"]
brk_diff = group_mix["Yankees"]["brk"] - group_mix["Other Orgs"]["brk"]
print(f"\nYankees hitters see {fb_diff:+.1%} fastballs and {brk_diff:+.1%} breaking balls")
print("relative to the comparison prospects. This could be a Yankee Stadium effect —")
print("pitchers go heater-heavy vs RHH at the short porch. But it also means Yankees")
print("hitters get LESS practice handling breaking stuff in-game — compounding the")
print("development gap.")


## 4. The Systemic Pattern

The pattern across Yankees prospects:

| Player | MiLB Discipline | MLB Result |
|--------|----------------|------------|
| Anthony Volpe | "Best Strike-Zone Discipline" (BA), .423 OBP in A-ball | Chase rate doubled (15% → 31%), seasonal regression pattern |
| Jasson Dominguez | "Best Strike-Zone Discipline" (BA), 17.3% K rate in AA/AAA | Chase rate nearly doubled (18% → 31%), breaking ball chase 41% |
| Oswald Peraza | Solid AAA numbers (.316 OBP) | 30% chase rate, 35% breaking ball chase, couldn't stick |

The exception is **Austin Wells** — 26.5% chase rate, 24.6% breaking ball chase. He's the one Yankees prospect who translated his discipline. Notably, he's a catcher — he may have benefited from seeing pitching from behind the plate, giving him an inherent advantage in pitch recognition.

## 5. What Other Orgs Do Differently

The Orioles (Henderson), Diamondbacks (Carroll), and Reds (De La Cruz) have all produced hitters who translated minor league discipline to the majors. Possible explanations:

1. **Better pitch-recognition training technology** — VR systems like [pitch recognition tools] that simulate MLB pitch tunnels and movement
2. **MiLB pitch quality development** — Coaching minor league *pitchers* to throw better breaking stuff, which incidentally develops better *hitters* in the same system
3. **Statcast-based readiness criteria** — Promoting based on approach metrics, not results
4. **Longer development timelines** — Not rushing prospects to fill MLB roster holes

The Yankees have historically prioritized developing pitching (Cole, Severino) and acquiring position players via trade/FA. Their hitting development pipeline has consistently failed to produce impact bats from within the system. Volpe and Dominguez are the latest examples of a decade-long pattern.

## 5. The Counterfactual: How Often Do Top Prospects Disappoint League-Wide?

Before concluding the Yankees have a systemic problem, we need a base rate.
If 40% of top-15 prospects disappoint across *all* orgs, then the Yankees
aren't an outlier — they're just unlucky. If the rate is much lower, the
pattern is meaningful.


In [ ]:
from fire_fishman.data.prospects import PROSPECT_DATA

# Base rate: what % of top-15 prospects are disappointing or bust?
top15 = [e for e in PROSPECT_DATA if e[3] <= 15]
non_yankees_top15 = [e for e in top15 if e[0] not in
                     ['Anthony Volpe', 'Jasson Dominguez', 'Ben Rice']]
yankees_top15 = [e for e in top15 if e[0] in
                 ['Anthony Volpe', 'Jasson Dominguez']]

print('PROSPECT OUTCOME BASE RATES (2019-2024 debuts)')
print('=' * 65)
print(f'\nAll top-15 prospects in cohort: n={len(top15)}')
for outcome in ['star', 'solid', 'disappointing', 'bust']:
    count = sum(1 for e in top15 if e[4] == outcome)
    pct = count / len(top15)
    print(f'  {outcome:<15} {count:>2}  ({pct:.0%})')

league_disappoint = sum(1 for e in non_yankees_top15
                        if e[4] in ('disappointing', 'bust'))
league_total = len(non_yankees_top15)
league_rate = league_disappoint / league_total

yankees_disappoint = sum(1 for e in yankees_top15
                         if e[4] in ('disappointing', 'bust'))
yankees_total = len(yankees_top15)
yankees_rate = yankees_disappoint / yankees_total

print(f'\nDisappointing + bust rate:')
print(f'  League-wide (excl NYY): {league_disappoint}/{league_total} = {league_rate:.0%}')
print(f'  Yankees top-15:         {yankees_disappoint}/{yankees_total} = {yankees_rate:.0%}')

print(f'\nInterpretation:')
print(f'  {league_rate:.0%} of non-Yankees top-15 prospects disappointed or busted.')
print(f'  Prospect failure is common league-wide — the Yankees are not a dramatic')
print(f'  outlier on outcome rate alone. The systemic argument rests not on')
print(f'  "more busts" but on the *pattern*: MiLB discipline awards followed')
print(f'  by identical MLB collapse across multiple prospects, suggesting a')
print(f'  development pipeline gap rather than bad luck.')
print(f'\n  Caveat: n={len(top15)} is small. These are suggestive rates, not proof.')


## 5b. The Killer Chart: Org Development Success Rates

Which organizations actually convert their prospects into productive MLB players?
Success rate = (star + solid outcomes) / total prospects tracked.


In [ ]:
# === THE KILLER CHART: Org-level development success rates ===
summary = get_org_summary()
target = summary[summary['org'].isin(TARGET_ORGS)].copy()
target = target.sort_values('success_rate', ascending=True)

MIN_N = 4  # minimum prospects for a reliable rate

print('PROSPECT DEVELOPMENT SUCCESS RATE BY ORGANIZATION')
print('=' * 65)
print(f'{"Org":<6} {"n":>3} {"Stars":>6} {"Solid":>6} {"Disapp":>7} {"Bust":>5} {"Success":>8}')
print('-' * 65)
for _, row in target.sort_values('success_rate', ascending=False).iterrows():
    flag = ' *' if row['n_prospects'] < MIN_N else ''
    print(f'{row["org"]:<6} {int(row["n_prospects"]):>3} '
          f'{int(row.get("star", 0)):>6} {int(row.get("solid", 0)):>6} '
          f'{int(row.get("disappointing", 0)):>7} {int(row.get("bust", 0)):>5} '
          f'{row["success_rate"]:>7.0%}{flag}')
print(f'\n* n < {MIN_N}: treat with caution')

# === Bar chart ===
fig, ax = plt.subplots(figsize=(10, 6))

colors = [org_colors.get(org, '#888888') for org in target['org']]
alphas = [1.0 if n >= MIN_N else 0.45 for n in target['n_prospects']]
bars = ax.barh(target['org'], target['success_rate'], color=colors, edgecolor='white', height=0.6)
for bar, alpha in zip(bars, alphas):
    bar.set_alpha(alpha)

# Annotate with n= and percentage
for bar, (_, row) in zip(bars, target.iterrows()):
    width = bar.get_width()
    label = f'{row["success_rate"]:.0%} (n={int(row["n_prospects"])})'
    if row['n_prospects'] < MIN_N:
        label += ' *'
    ax.text(width + 0.02, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1.15)
ax.set_xlabel('Success Rate (star + solid / total)', fontsize=12)
ax.set_title('Prospect Development Success Rate by Organization\n'
             '2019-2024 debuts, top-100 ranked prospects',
             fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.axvline(x=summary['success_rate'].mean(), color='gray', linestyle='--', alpha=0.5,
           label=f'All-org avg: {summary["success_rate"].mean():.0%}')
ax.legend(loc='lower right')
ax.annotate(f'* n < {MIN_N}: small sample, treat with caution',
            xy=(0.98, 0.02), xycoords='axes fraction', ha='right', fontsize=9,
            fontstyle='italic', color='gray')

plt.tight_layout()
plt.savefig('../outputs/figures/org_success_rates.png', dpi=150, bbox_inches='tight')
plt.show()

### Discipline Heatmap: What Do Elite Development Orgs Do Differently?


In [ ]:
# === Discipline metrics heatmap by org ===
org_avgs = org_metrics.groupby('org')[metric_cols].mean()
org_avgs = org_avgs.loc[[o for o in TARGET_ORGS if o in org_avgs.index]]

# Rename columns for display
display_cols = {
    'chase_rate': 'Chase Rate',
    'brk_chase': 'Breaking Chase',
    'off_chase': 'Offspeed Chase',
    'whiff_rate': 'Whiff Rate',
    'zone_contact': 'Zone Contact',
}
heatmap_df = org_avgs.rename(columns=display_cols)

fig, ax = plt.subplots(figsize=(10, 5))

# For chase/whiff metrics, lower is better (red = bad).
# For zone contact, higher is better.
# Use a custom annotation to show values as percentages.
sns.heatmap(heatmap_df, annot=True, fmt='.1%', cmap='RdYlGn_r',
            linewidths=1, linecolor='white', ax=ax,
            cbar_kws={'label': 'Rate (lower chase/whiff = better, higher contact = better)'})

ax.set_title('MLB Plate Discipline by Development Organization\n'
             'Avg metrics for prospects developed by each org',
             fontsize=13, fontweight='bold')
ax.set_ylabel('')

# Add n= to org labels
org_n = org_metrics.groupby('org').size()
new_labels = [f'{org} (n={org_n.get(org, 0)})' for org in heatmap_df.index]
ax.set_yticklabels(new_labels, rotation=0)

plt.tight_layout()
plt.savefig('../outputs/figures/org_discipline_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNote: These are average metrics across each org\'s prospect cohort.')
print('Sample sizes vary (n=2 to n=6). Interpret org-level differences')
print('directionally, not as proof of development superiority.')


## 6. Conclusion: The Pattern Points to Process, Not Only Players

The data is clear:

- Both Volpe and Dominguez had **elite, measurable discipline** in the minors
- Both regressed against MLB pitching in the **same specific ways** (chase rate on breaking balls and offspeed)
- The regression follows a **repeating seasonal pattern** (Volpe) or is **immediately exploited by the league** (Dominguez)
- Other Yankees system products (Peraza) show the same pattern
- Prospects from other organizations translate their discipline at much higher rates

This isn't a scouting failure — the scouts were right about the discipline. It's a **development failure**: the Yankees' MiLB system doesn't bridge the pitch quality gap between AAA and MLB. Their hitters arrive at the major league level with minor league calibration, and they're expected to adjust on the fly.

The fix isn't finding better prospects. It's building a development pipeline that actually prepares them for what they'll face.